In [1]:
print("Let's Start")

Let's Start


In [2]:
import re
import string
import automac
import mlflow
import mlflow.sklearn
import dagshub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score 
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [6]:
import xgboost
import lightgbm
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

In [15]:
df = pd.read_csv('/mnt/WindowsD/Artificial Intelligence/Datasets/tweets_emotions.csv' , index_col=0).drop(columns='tweet_id')
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [16]:
x=(df['sentiment']=='happiness') | (df['sentiment'] == 'sadness')
df = df[x]
df['sentiment'] =df['sentiment'].replace({'sadness' : 0 , 'happiness' : 1}) 
x_ = df['content'].apply(automac.text.transform)
y = df['sentiment']
y = pd.to_numeric(y, errors='raise').astype(int)


In [14]:
x_

Series([], Name: content, dtype: str)

In [ ]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='mlops-mini-project', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/mlops-mini-project.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('BOW vs TFIDF')

2026/11/14 10:55:13 INFO mlflow.tracking.fluent: Experiment with name 'BOW vs TFIDF' does not exist. Creating a new experiment.


<Experiment: artifact_location=('/mnt/WindowsD/Artificial '
 'Intelligence/MLOPS/projects/mlops-mini-project/notebooks/mlruns/2'), creation_time=1794633913613, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1794633913613, lifecycle_stage='active', name='BOW vs TFIDF', tags={}, trace_location=None, workspace='default'>

In [7]:
vectorizers = {
    'bow' : CountVectorizer(),
    'tfidf' : TfidfVectorizer()

}

algorithms = {
    'LogisticRegression' : LogisticRegression(),
    'RandomForest' : RandomForestClassifier(),
    'NaiveBayes' : MultinomialNB(),
    'Xgboost' : xgboost.XGBClassifier(),
    'Lightgbm' : lightgbm.LGBMClassifier()
}

In [ ]:
from sklearn.tree import _tree

with mlflow.start_run(run_name = 'All Experiments') as parent:
    for algo_name , algorithm in algorithms.items():
        for vect , vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vect}" , nested=True) as child:
                x =vectorizer.fit_transform(x_)

                x_train , x_test , y_train , y_test= train_test_split(x , y , test_size=0.2 , random_state=0)

                mlflow.log_param("vectorizer" , vect)
                mlflow.log_param("algorithm" , algo_name)
                mlflow.log_param("test_size" , 0.2)

                model = algorithm
                model.fit(x_train , y_train)

                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)


                y_pred = model.predict(x_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)

                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)


                mlflow.sklearn.log_model(
                    model,
                    artifact_path="model",
                    skops_trusted_types=["sklearn.tree._tree.Tree"],
                )
            

                # Save and log the notebook
                mlflow.log_artifact("2_bowVStfidf.ipynb")
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vect}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")


2026/11/14 11:38:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 11:39:02 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: LogisticRegression, Feature Engineering: bow
Accuracy: 0.7942168674698795
Precision: 0.7961538461538461
Recall: 0.7938638542665388
F1 Score: 0.7950072011521844


2026/11/14 11:39:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 11:39:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Algorithm: LogisticRegression, Feature Engineering: tfidf
Accuracy: 0.8067469879518072
Precision: 0.8086538461538462
Recall: 0.8063279002876318
F1 Score: 0.8074891982717235


2026/11/14 11:40:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
